In [9]:
import altair as alt
import pandas as pd

# 1. Cargar y limpiar datos
df = pd.read_csv("01-base-de-datos-natalidad-limpia.csv", sep=";", skiprows=1)
df = df.dropna(subset=["Country Name"])

# Formatear columnas de años
cols_anios = [c for c in df.columns if c != "Country Name"]
df = df.rename(columns={c: c.replace("-", "").strip() for c in cols_anios})

# Pasar a formato largo
df_long = df.melt(
    id_vars=["Country Name"], var_name="Año", value_name="Tasa_Natalidad"
)
df_long["Año"] = df_long["Año"].astype(int)
df_long["Tasa_Natalidad"] = (
    df_long["Tasa_Natalidad"].astype(str).str.replace(",", ".").astype(float)
)

# Cambiar nombres de países
mapa_paises = {
    "Arabia Saudita": "Arabia Saudita",
    "Australia": "Australia",
    "Estados Unidos": "Estados Unidos",
    "Francia": "Francia",
    "Irán, República Islámica del": "Irán",
    "Israel": "Israel",
    "Jordania": "Jordania",
    "Líbano": "Líbano",
    "México": "México",
    "Noruega": "Noruega",
    "República Árabe Siria": "Siria",
    "Ribera Occidental y Gaza": "Palestina (Gaza y Ribera Occ.)",
}
df_long["Pais"] = df_long["Country Name"].map(mapa_paises)

# Guardar base procesada
df_long[["Pais", "Año", "Tasa_Natalidad"]].to_csv(
    "datos_procesados.csv", index=False
)

# 2. Generar gráfico interactivo en Altair
highlight = alt.selection_point(fields=["Pais"], bind="legend")

chart = (
    alt.Chart(df_long)
    .mark_line(point=True)
    .encode(
        x=alt.X("Año:O", title="Año"),
        y=alt.Y(
            "Tasa_Natalidad:Q",
            title="Tasa bruta de natalidad (por 1.000 habitantes)",
        ),
        color=alt.Color(
            "Pais:N",
            title="País / Territorio",
            scale=alt.Scale(scheme="category10"),
        ),
        opacity=alt.condition(highlight, alt.value(1.0), alt.value(0.2)),
        strokeWidth=alt.condition(
            alt.datum.Pais == "Palestina (Gaza y Ribera Occ.)",
            alt.value(3.5),
            alt.value(1.5),
        ),
        tooltip=["Pais", "Año", "Tasa_Natalidad"],
    )
    .add_params(highlight)
    .properties(
        width=700,
        height=400,
        title="Evolución de la Tasa Bruta de Natalidad (1998 - 2024)",
    )
)

# 3. Exportar archivos
chart.save("visualizacion.html")

chart


alt.Chart(...)